## Solución Taller 04- Optimización de Redes Metabólicas

In [31]:
# Librerías a utilizar en el taller
import numpy as np
import pandas as pd
import cobra
from cobra.flux_analysis.variability import flux_variability_analysis

# Definir Gurobi como solver
cobra.core.configuration.Configuration(solver="gurobi");

In [32]:
# Cargar el modelo que se utilizará en el taller
model = cobra.io.read_sbml_model("iRBC283.xml")

### Pregunta a)

Se realiza un FVA para los exchanges, ocupando 0 como la fracción de velocidad máxima de crecimiento.

In [36]:
fva_result = flux_variability_analysis(model, fraction_of_optimum=0, reaction_list=model.exchanges)

In [38]:
min_fva  = fva_result['minimum']
max_fva  = fva_result['maximum']
ref_span = max_fva - min_fva

### Pregunta b)

En esta pregunta se deben aplicar las modificaciones al modelo y efectuar un FVA para evaluar el impacto en los límites de los exchanges. 

Para bloquear una reacción se debe definir nulo tanto el límite superior como inferior de la reacción.

In [42]:
# Función para realizar FVA después de cambiar límites
def simulate_snp(model, target_reactions):
    with model as temp_model:
        if isinstance(target_reactions, list):
            for rxn_id in target_reactions:
                rxn = temp_model.reactions.get_by_id(rxn_id)
                rxn.lower_bound = 0
                rxn.upper_bound = 0
        else:
            rxn = temp_model.reactions.get_by_id(target_reactions)
            rxn.lower_bound = 0
            rxn.upper_bound = 0
        fva_res = flux_variability_analysis(temp_model, fraction_of_optimum=0, reaction_list=temp_model.exchanges, processes=4)
    return fva_res

# Función para determinar exchanges impactados
def impacted_exchanges(min_ref, max_ref, min_mut, max_mut, threshold, exchanges):
    impacted = []
    for rxn in exchanges:
        if abs(min_ref[rxn] - min_mut[rxn]) > threshold[rxn] or abs(max_ref[rxn] - max_mut[rxn]) > threshold[rxn]:
            impacted.append(rxn)
    return impacted


Para cada SNPs se evalúa el impacto, ocupando un 40% del rango original de la reacción.

In [44]:
# Lista de reacciones de intercambio
exchange_ids = [rxn.id for rxn in model.exchanges]

# Análisis de impacto (criterio de cambio significativo > 40% del rango basal)
threshold = 0.4 * ref_span

# ADA impacto
ada_fva = simulate_snp(model, "ADA")
impacted_ada = impacted_exchanges(min_fva, max_fva, ada_fva["minimum"], ada_fva["maximum"], threshold, exchange_ids)
print("Exchanges impactados por ADA")
print(impacted_ada)

# FBA impacto
fba_fva = simulate_snp(model, "FBA")
impacted_fba = impacted_exchanges(min_fva, max_fva, fba_fva["minimum"], fba_fva["maximum"], threshold, exchange_ids)
print("\nExchanges impactados por FBA")
print(impacted_fba)

# NP impacto
np_reactions = ["NP1", "PNP", "PUNP1", "PUNP3", "PUNP5"]
np_fva = simulate_snp(model, np_reactions)
impacted_np = impacted_exchanges(min_fva, max_fva, np_fva["minimum"], np_fva["maximum"], threshold, exchange_ids)
print("\nExchanges impactados por NP pathway")
print(impacted_np)


Exchanges impactados por ADA
[]

Exchanges impactados por FBA
['EX_co2_e', 'EX_hco3_e', 'EX_lac__D_e']

Exchanges impactados por NP pathway
['EX_hxan_e', 'EX_nac_e', 'EX_ncam_e']


Se repite el proceso para cada efecto de las drogas

In [45]:
# ABCC4 impacto
abcc_reactions = ["CAMPt", "CGMPt"]
abcc_fva = simulate_snp(model, abcc_reactions)
impacted_abcc = impacted_exchanges(min_fva, max_fva, abcc_fva["minimum"], abcc_fva["maximum"], threshold, exchange_ids)
print("Exchanges impactados por ABCC4")
print(impacted_abcc)

# AHCY impacto
ahcy_fva = simulate_snp(model, "AHCi")
impacted_ahcy = impacted_exchanges(min_fva, max_fva, ahcy_fva["minimum"], ahcy_fva["maximum"], threshold, exchange_ids)
print("\nExchanges impactados por AHCY")
print(impacted_ahcy)

# CA impacto
ca_fva = simulate_snp(model, "HCO3E")
impacted_ca = impacted_exchanges(min_fva, max_fva, ca_fva["minimum"], ca_fva["maximum"], threshold, exchange_ids)
print("\nExchanges impactados por CA")
print(impacted_ca)


Exchanges impactados por ABCC4
['EX_35cgmp_e', 'EX_camp_e']

Exchanges impactados por AHCY
['EX_hcys__L_e', 'EX_met__L_e']

Exchanges impactados por CA
['EX_co2_e', 'EX_hco3_e']
